# Сверка: Excel (Jan–Jun) vs `MPOS_RENT.N_AMT`

Помесячное сравнение `commission_monthly`:
- **Lake:** `ods_alpha.scd1_mrc_pos_rent.n_amt` (зерно `c_nmrc`, период `d_rent`);
- **Excel:** отчёты `01_Январь` … `06_Июнь_2026.xlsx` (зерно `ИНН + ID договора`);
- **Ключ:** `inn_key + agr_id_key` после маппинга `c_nmrc -> agr_terms -> agreements -> companies`;
- **Правило:** если ключа нет в Озере — ожидаемая комиссия = 0.

Секции:
0. mapping coverage (no / unique / ambiguous);
1–5. % полного совпадения, TOP delta/pos/neg, Excel≠0 без Lake, rounding;
**4. расширенный QC:** ambiguous vs Excel, grain d_rent, gross/net, Excel dup, SA-контур, stability;
5. апрель right-only по тарифам;
6. выгрузка отчёта.

**3b.** диагностика января (header/колонка/масштаб), если Excel total аномальный.


In [ ]:
import re
import time
from calendar import monthrange
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


def first_nonempty_tariff(series):
    for v in series:
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s and s.lower() not in {'nan', 'none', 'null'}:
            return s
    return None


def month_bounds(month_label):
    y, m = map(int, month_label.split('-'))
    start = f'{y:04d}-{m:02d}-01'
    end = f'{y:04d}-{m:02d}-{monthrange(y, m)[1]:02d}'
    return start, end


## 0) Конфиг


In [ ]:
# === Таблицы (MPOS_RENT = scd1_mrc_pos_rent) ===
mrc_table = 'ods_alpha.scd1_mrc_pos_rent'
agr_terms_table = 'ods_alpha.scd1_agr_terms'
agreements_table = 'ods_alpha.scd1_agreements'
companies_table = 'ods_alpha.scd1_companies'

# === Excel Jan–Jun 2026 ===
excel_data_dir = Path('/home/jovyan/documents/Equaring/Data')
excel_header_default = 0
excel_header_by_month = {
    '2026-01': 1,  # как в commission_monthly_n_amt_dq_checks
}
excel_reference_by_month = {
    '2026-01': str(excel_data_dir / '01_Январь_2026.xlsx'),
    '2026-02': str(excel_data_dir / '02_Февраль_2026.xlsx'),
    '2026-03': str(excel_data_dir / '03_Март_2026.xlsx'),
    '2026-04': str(excel_data_dir / '04_Апрель_2026.xlsx'),
    '2026-05': str(excel_data_dir / '05_Май_2026.xlsx'),
    '2026-06': str(excel_data_dir / '06_Июнь_2026.xlsx'),
}

# === Пороги match ===
exact_abs_tol = 0.01
near_abs_tol = 1.0
near_pct_tol = 1.0  # %
top_n = 10

# === Подключение ===
impala_db = 'sandbox_ai'
impala_mem_limit = '8g'
impala_user_name = 'Shestopalov-VYur'

# === Выгрузка ===
output_dir = excel_data_dir
output_report_path = output_dir / 'mpos_rent_excel_compare_all_months.xlsx'

print('months:', list(excel_reference_by_month))
print('mrc_table:', mrc_table)
for m, p in excel_reference_by_month.items():
    h = excel_header_by_month.get(m, excel_header_default)
    print(f'  {m}: {p} (header={h})')
print('output:', output_report_path)


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': impala_db},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': impala_user_name}
)
imp._init_connection()


def run_sql(sql_text, step_name='query', mem_limit=impala_mem_limit):
    start_ts = time.perf_counter()
    print(f'[{step_name}] start')
    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        df = imp.fetch(sql_text)
    elapsed = round(time.perf_counter() - start_ts, 2)
    rows = len(df) if isinstance(df, pd.DataFrame) else 0
    print(f'[{step_name}] done in {elapsed}s, rows={rows:,}')
    return df


print('Impala connection initialized')


## 1) Доступ к таблице


In [ ]:
sql_access = f"""
select 1 as probe_ok
from {mrc_table}
limit 1
"""

access_ok = False
access_error = None

try:
    access_df = run_sql(sql_access, step_name='access_check')
    access_ok = True
    print('ACCESS_OK')
    display(access_df)
except Exception as exc:
    access_error = f'{type(exc).__name__}: {exc}'
    print('ACCESS_ERROR:', access_error)

display(pd.DataFrame([{'table': mrc_table, 'access_ok': access_ok, 'error': access_error}]))


## 2) Функции сверки месяца (Lake + Excel + 4 блока)


In [ ]:
excel_col_map = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'comm_monthly_col': [
        'Комиссия в месяц',
        'Комиссия CN (₽ в месяц)',
        'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)',
        'Комиссия (руб в месяц)',
    ],
    'tariff_col': ['Тариф', 'Тарифный план', 'tariff_name'],
}


def build_mapping_cte(month_start, month_end):
    return f"""
with rent_base as (
    select
        cast(c_nmrc as string) as c_nmrc,
        cast(d_rent as date) as d_rent_dt,
        cast(n_amt as double) as n_amt_num,
        cast(ods_commit_ts as timestamp) as ods_commit_ts,
        cast(ods_insert_ts as timestamp) as ods_insert_ts,
        cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
        coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
    from {mrc_table}
    where c_nmrc is not null
      and cast(d_rent as date) between cast('{month_start}' as date) and cast('{month_end}' as date)
), rent_ranked as (
    select
        *,
        row_number() over (
            partition by c_nmrc, d_rent_dt
            order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
        ) as rn
    from rent_base
    where ods_deleted_flg not in ('1', 'Y', 'y')
), rent_dedup as (
    select c_nmrc, d_rent_dt, n_amt_num
    from rent_ranked
    where rn = 1
), terms_active as (
    select distinct
        cast(t.n_agr as string) as n_agr,
        cast(t.c_nmrc as string) as c_nmrc,
        cast(t.d_valid_from as date) as d_valid_from,
        cast(t.d_valid_to as date) as d_valid_to
    from {agr_terms_table} t
    where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
      and t.c_nmrc is not null
      and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
      and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
), agreements_active as (
    select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_cmp_client as string) as n_cmp_client,
        cast(a.d_valid_from as date) as d_valid_from,
        cast(a.d_valid_to as date) as d_valid_to
    from {agreements_table} a
    where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
      and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
), companies_active as (
    select distinct
        cast(c.n_cmp as string) as n_cmp,
        regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
    from {companies_table} c
    where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
), mapped_raw as (
    select
        r.c_nmrc,
        r.d_rent_dt,
        r.n_amt_num,
        c.inn_key,
        cast(a.agr_id as string) as agr_id_key
    from rent_dedup r
    left join terms_active t
      on t.c_nmrc = r.c_nmrc
     and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
    left join agreements_active a
      on a.n_agr = t.n_agr
     and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
    left join companies_active c
      on c.n_cmp = a.n_cmp_client
)
"""


def load_lake_month(month_label, month_start, month_end):
    cte = build_mapping_cte(month_start, month_end)
    sql = f"""
    {cte}
    , map_stats as (
        select
            c_nmrc,
            d_rent_dt,
            n_amt_num,
            count(distinct case when inn_key is not null and agr_id_key is not null
                then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
        from mapped_raw
        group by c_nmrc, d_rent_dt, n_amt_num
    ), mapped_unique as (
        select
            mr.c_nmrc,
            mr.d_rent_dt,
            mr.n_amt_num,
            max(mr.inn_key) as inn_key,
            max(mr.agr_id_key) as agr_id_key
        from mapped_raw mr
        join map_stats ms
          on ms.c_nmrc = mr.c_nmrc
         and ms.d_rent_dt = mr.d_rent_dt
         and ms.n_amt_num = mr.n_amt_num
        where ms.valid_key_cnt = 1
          and mr.inn_key is not null
          and mr.agr_id_key is not null
        group by mr.c_nmrc, mr.d_rent_dt, mr.n_amt_num
    )
    select
        '{month_label}' as month_label,
        inn_key,
        agr_id_key,
        sum(n_amt_num) as commission_monthly_lake
    from mapped_unique
    group by inn_key, agr_id_key
    """
    df = run_sql(sql, step_name=f'lake_{month_label}')
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_lake'])
    df['inn_key'] = df['inn_key'].apply(normalize_inn_q1)
    df['agr_id_key'] = df['agr_id_key'].apply(normalize_agr_q1)
    df['commission_monthly_lake'] = pd.to_numeric(df['commission_monthly_lake'], errors='coerce')
    return (
        df.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['month_label', 'inn_key', 'agr_id_key'], as_index=False)
        .agg(commission_monthly_lake=('commission_monthly_lake', 'sum'))
    )


def load_excel_month(month_label, excel_path, excel_header):
    ex_raw = pd.read_excel(excel_path, header=excel_header)
    resolved = {k: pick_col_robust(ex_raw.columns, v) for k, v in excel_col_map.items()}
    missing = [k for k, v in resolved.items() if v is None and k != 'tariff_col']
    if missing:
        raise ValueError(f'[{month_label}] Не найдены колонки Excel: {missing}. Доступные: {list(ex_raw.columns)}')

    ex = ex_raw.copy()
    ex['inn_key'] = ex[resolved['inn_col']].apply(normalize_inn_q1)
    ex['agr_id_key'] = ex[resolved['agr_col']].apply(normalize_agr_q1)
    ex['commission_monthly_excel'] = to_num_series(ex[resolved['comm_monthly_col']])
    if resolved.get('tariff_col') is not None:
        ex['tariff_name_excel'] = ex[resolved['tariff_col']]
    else:
        ex['tariff_name_excel'] = None
    ex['month_label'] = month_label

    agg = (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['month_label', 'inn_key', 'agr_id_key'], as_index=False)
        .agg(
            commission_monthly_excel=('commission_monthly_excel', 'max'),
            tariff_name_excel=('tariff_name_excel', first_nonempty_tariff),
        )
    )
    print(f'[{month_label}] excel keys={len(agg):,}; cols={resolved}')
    return agg, resolved



def load_mapping_coverage_month(month_label, month_start, month_end):
    cte = build_mapping_cte(month_start, month_end)
    sql = f"""
    {cte}
    , map_stats as (
        select
            c_nmrc,
            d_rent_dt,
            n_amt_num,
            count(distinct case when inn_key is not null and agr_id_key is not null
                then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
        from mapped_raw
        group by c_nmrc, d_rent_dt, n_amt_num
    )
    select
        '{month_label}' as month_label,
        count(*) as rent_rows_after_dedup,
        coalesce(sum(n_amt_num), 0) as total_n_amt,
        sum(case when valid_key_cnt = 0 then 1 else 0 end) as no_mapping_rows,
        coalesce(sum(case when valid_key_cnt = 0 then n_amt_num else 0 end), 0) as no_mapping_n_amt,
        sum(case when valid_key_cnt = 1 then 1 else 0 end) as unique_mapping_rows,
        coalesce(sum(case when valid_key_cnt = 1 then n_amt_num else 0 end), 0) as unique_mapping_n_amt,
        sum(case when valid_key_cnt > 1 then 1 else 0 end) as ambiguous_mapping_rows,
        coalesce(sum(case when valid_key_cnt > 1 then n_amt_num else 0 end), 0) as ambiguous_mapping_n_amt
    from map_stats
    """
    df = run_sql(sql, step_name=f'mapping_coverage_{month_label}')
    if df is None or len(df) == 0:
        return pd.DataFrame([{
            'month_label': month_label,
            'rent_rows_after_dedup': 0,
            'total_n_amt': 0.0,
            'no_mapping_rows': 0,
            'no_mapping_n_amt': 0.0,
            'unique_mapping_rows': 0,
            'unique_mapping_n_amt': 0.0,
            'ambiguous_mapping_rows': 0,
            'ambiguous_mapping_n_amt': 0.0,
            'no_mapping_rows_pct': np.nan,
            'unique_mapping_rows_pct': np.nan,
            'ambiguous_mapping_rows_pct': np.nan,
            'no_mapping_n_amt_pct': np.nan,
            'unique_mapping_n_amt_pct': np.nan,
            'ambiguous_mapping_n_amt_pct': np.nan,
        }])

    for col in [
        'rent_rows_after_dedup', 'total_n_amt',
        'no_mapping_rows', 'no_mapping_n_amt',
        'unique_mapping_rows', 'unique_mapping_n_amt',
        'ambiguous_mapping_rows', 'ambiguous_mapping_n_amt',
    ]:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    total_rows = float(df.loc[0, 'rent_rows_after_dedup'])
    total_amt = float(df.loc[0, 'total_n_amt'])
    if total_rows:
        df['no_mapping_rows_pct'] = df['no_mapping_rows'] / total_rows * 100.0
        df['unique_mapping_rows_pct'] = df['unique_mapping_rows'] / total_rows * 100.0
        df['ambiguous_mapping_rows_pct'] = df['ambiguous_mapping_rows'] / total_rows * 100.0
    else:
        df['no_mapping_rows_pct'] = np.nan
        df['unique_mapping_rows_pct'] = np.nan
        df['ambiguous_mapping_rows_pct'] = np.nan

    if total_amt:
        df['no_mapping_n_amt_pct'] = df['no_mapping_n_amt'] / total_amt * 100.0
        df['unique_mapping_n_amt_pct'] = df['unique_mapping_n_amt'] / total_amt * 100.0
        df['ambiguous_mapping_n_amt_pct'] = df['ambiguous_mapping_n_amt'] / total_amt * 100.0
    else:
        df['no_mapping_n_amt_pct'] = np.nan
        df['unique_mapping_n_amt_pct'] = np.nan
        df['ambiguous_mapping_n_amt_pct'] = np.nan
    return df


def load_mapping_ambiguous_top_month(month_label, month_start, month_end, limit=50):
    cte = build_mapping_cte(month_start, month_end)
    sql = f"""
    {cte}
    , map_stats as (
        select
            c_nmrc,
            d_rent_dt,
            n_amt_num,
            count(distinct case when inn_key is not null and agr_id_key is not null
                then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
        from mapped_raw
        group by c_nmrc, d_rent_dt, n_amt_num
    )
    select
        '{month_label}' as month_label,
        c_nmrc,
        d_rent_dt,
        n_amt_num,
        valid_key_cnt
    from map_stats
    where valid_key_cnt > 1
    order by n_amt_num desc, valid_key_cnt desc, d_rent_dt desc, c_nmrc
    limit {int(limit)}
    """
    df = run_sql(sql, step_name=f'mapping_ambiguous_top_{month_label}')
    if df is None:
        return pd.DataFrame(columns=['month_label', 'c_nmrc', 'd_rent_dt', 'n_amt_num', 'valid_key_cnt'])
    return df


def _classify_mismatch_reason(row):
    lake = float(row['commission_monthly_lake'])
    excel = float(row['commission_monthly_excel'])
    delta = float(row['delta_abs'])
    abs_d = abs(delta)
    status = row['merge_status']

    if abs_d < exact_abs_tol:
        return 'exact_match'

    # Likely rounding / kopecks
    if abs_d < near_abs_tol:
        return 'near_rounding_abs_lt_1'

    # Relative near
    if excel != 0 and abs_d / abs(excel) * 100.0 < near_pct_tol:
        return 'near_rounding_pct_lt_1'

    # Format / scale suspects: ~100x or ~0.01x
    if excel != 0 and lake != 0:
        ratio = lake / excel
        if 90 <= abs(ratio) <= 110 or 0.009 <= abs(ratio) <= 0.011:
            return 'suspect_scale_100x_or_0_01x'

    # Comma-as-dot style: values equal if ignore decimals oddly — rare; flag if almost equal at integer
    if abs(round(lake) - round(excel)) < exact_abs_tol and abs_d >= exact_abs_tol:
        return 'suspect_integer_round'

    if status == 'right_only' and abs(excel) >= exact_abs_tol:
        return 'excel_nz_missing_in_lake'
    if status == 'left_only' and abs(lake) >= exact_abs_tol:
        return 'lake_nz_missing_in_excel'
    if abs(excel) < exact_abs_tol and abs(lake) >= exact_abs_tol:
        return 'lake_nz_excel_zero'
    if abs(lake) < exact_abs_tol and abs(excel) >= exact_abs_tol:
        return 'excel_nz_lake_zero'
    if delta > 0:
        return 'lake_gt_excel'
    return 'lake_lt_excel'


def compare_month(month_label, excel_path, excel_header):
    month_start, month_end = month_bounds(month_label)
    lake_df = load_lake_month(month_label, month_start, month_end)
    excel_df, resolved = load_excel_month(month_label, excel_path, excel_header)

    compare_df = lake_df.merge(
        excel_df[['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_excel', 'tariff_name_excel']],
        on=['month_label', 'inn_key', 'agr_id_key'],
        how='outer',
        indicator=True,
    ).rename(columns={'_merge': 'merge_status'})

    # Правило: если ключа нет в Озере — ожидаемая комиссия Lake = 0
    compare_df['in_lake'] = compare_df['merge_status'].isin(['both', 'left_only'])
    compare_df['in_excel'] = compare_df['merge_status'].isin(['both', 'right_only'])
    compare_df['commission_monthly_lake'] = compare_df['commission_monthly_lake'].fillna(0.0)
    compare_df['commission_monthly_excel'] = compare_df['commission_monthly_excel'].fillna(0.0)
    compare_df['expected_lake'] = np.where(compare_df['in_lake'], compare_df['commission_monthly_lake'], 0.0)

    compare_df['delta_abs'] = compare_df['expected_lake'] - compare_df['commission_monthly_excel']
    compare_df['delta_pct'] = np.where(
        compare_df['commission_monthly_excel'] != 0,
        compare_df['delta_abs'].abs() / compare_df['commission_monthly_excel'].abs() * 100.0,
        np.nan,
    )
    compare_df['is_exact_match'] = compare_df['delta_abs'].abs() < exact_abs_tol
    compare_df['is_near_match'] = (
        (compare_df['delta_abs'].abs() < near_abs_tol)
        | (compare_df['delta_pct'].fillna(np.inf) < near_pct_tol)
    )
    compare_df['mismatch_reason'] = compare_df.apply(_classify_mismatch_reason, axis=1)

    # Universe for "полное совпадение": все ключи Excel (+ left_only lake-only keys тоже в outer)
    # Бизнес-фокус: % совпадения на всех Excel-ключах (нет в Lake => Excel должен быть 0)
    excel_universe = compare_df[compare_df['in_excel']].copy()
    excel_univ_n = int(len(excel_universe))
    excel_exact_n = int(excel_universe['is_exact_match'].sum()) if excel_univ_n else 0
    excel_near_n = int(excel_universe['is_near_match'].sum()) if excel_univ_n else 0

    all_n = int(len(compare_df))
    all_exact_n = int(compare_df['is_exact_match'].sum()) if all_n else 0

    intersection = compare_df[compare_df['merge_status'] == 'both'].copy()
    inter_n = int(len(intersection))
    exact_n = int(intersection['is_exact_match'].sum()) if inter_n else 0
    near_n = int(intersection['is_near_match'].sum()) if inter_n else 0

    excel_total = float(excel_df['commission_monthly_excel'].fillna(0).sum()) if len(excel_df) else 0.0
    lake_total = float(lake_df['commission_monthly_lake'].fillna(0).sum()) if len(lake_df) else 0.0
    inter_excel = float(intersection['commission_monthly_excel'].fillna(0).sum()) if inter_n else 0.0
    inter_lake = float(intersection['commission_monthly_lake'].fillna(0).sum()) if inter_n else 0.0

    # Компенсация дельт: gross+/gross-/net на всех ключах и на Excel-universe
    mismatches = compare_df[~compare_df['is_exact_match']].copy()
    pos_delta_sum = float(mismatches.loc[mismatches['delta_abs'] > 0, 'delta_abs'].sum()) if len(mismatches) else 0.0
    neg_delta_sum = float(mismatches.loc[mismatches['delta_abs'] < 0, 'delta_abs'].sum()) if len(mismatches) else 0.0
    net_delta = float(compare_df['delta_abs'].sum()) if len(compare_df) else 0.0

    excel_mismatches = excel_universe[~excel_universe['is_exact_match']].copy()
    excel_pos_delta_sum = float(excel_mismatches.loc[excel_mismatches['delta_abs'] > 0, 'delta_abs'].sum()) if len(excel_mismatches) else 0.0
    excel_neg_delta_sum = float(excel_mismatches.loc[excel_mismatches['delta_abs'] < 0, 'delta_abs'].sum()) if len(excel_mismatches) else 0.0

    reason_stats_df = (
        compare_df.groupby('mismatch_reason', as_index=False)
        .agg(
            key_cnt=('agr_id_key', 'count'),
            delta_sum=('delta_abs', 'sum'),
            abs_delta_sum=('delta_abs', lambda s: s.abs().sum()),
            excel_sum=('commission_monthly_excel', 'sum'),
            lake_sum=('commission_monthly_lake', 'sum'),
        )
        .sort_values('abs_delta_sum', ascending=False)
        .reset_index(drop=True)
    )
    reason_stats_df['month_label'] = month_label

    summary_df = pd.DataFrame([{
        'month_label': month_label,
        'month_start': month_start,
        'month_end': month_end,
        'excel_key_cnt': int(len(excel_df)),
        'lake_key_cnt': int(len(lake_df)),
        'intersection_key_cnt': inter_n,
        'only_excel_key_cnt': int((compare_df['merge_status'] == 'right_only').sum()),
        'only_lake_key_cnt': int((compare_df['merge_status'] == 'left_only').sum()),
        # Полное совпадение: правило "нет в Lake => Excel=0", знаменатель = все Excel-ключи
        'full_exact_match_cnt': excel_exact_n,
        'full_exact_match_rate_pct': excel_exact_n / excel_univ_n * 100.0 if excel_univ_n else np.nan,
        'full_near_match_cnt': excel_near_n,
        'full_near_match_rate_pct': excel_near_n / excel_univ_n * 100.0 if excel_univ_n else np.nan,
        'full_mismatch_cnt': excel_univ_n - excel_exact_n,
        # Старые метрики на пересечении (оба источника)
        'intersection_exact_match_cnt': exact_n,
        'intersection_exact_match_rate_pct': exact_n / inter_n * 100.0 if inter_n else np.nan,
        'intersection_near_match_rate_pct': near_n / inter_n * 100.0 if inter_n else np.nan,
        'excel_total': excel_total,
        'lake_total': lake_total,
        'total_delta_abs': lake_total - excel_total,
        'total_delta_pct': abs(lake_total - excel_total) / abs(excel_total) * 100.0 if excel_total else np.nan,
        # Компенсация: почему net << max(|single delta|)
        'mismatch_pos_delta_sum': pos_delta_sum,
        'mismatch_neg_delta_sum': neg_delta_sum,
        'mismatch_net_delta': net_delta,
        'excel_univ_pos_delta_sum': excel_pos_delta_sum,
        'excel_univ_neg_delta_sum': excel_neg_delta_sum,
        'excel_univ_net_delta': excel_pos_delta_sum + excel_neg_delta_sum,
        'intersection_excel_total': inter_excel,
        'intersection_lake_total': inter_lake,
        'intersection_delta_abs': inter_lake - inter_excel,
        'all_keys_exact_match_rate_pct': all_exact_n / all_n * 100.0 if all_n else np.nan,
    }])

    cols_ex = [
        'month_label', 'inn_key', 'agr_id_key', 'merge_status',
        'commission_monthly_lake', 'commission_monthly_excel',
        'expected_lake', 'delta_abs', 'delta_pct', 'mismatch_reason', 'tariff_name_excel',
    ]

    top_delta_df = (
        mismatches.sort_values(by='delta_abs', key=lambda s: s.abs(), ascending=False)
        .head(top_n)[cols_ex]
        .reset_index(drop=True)
    )

    top_pos_delta_df = (
        mismatches[mismatches['delta_abs'] > 0]
        .sort_values('delta_abs', ascending=False)
        .head(top_n)[cols_ex]
        .reset_index(drop=True)
    )
    top_neg_delta_df = (
        mismatches[mismatches['delta_abs'] < 0]
        .sort_values('delta_abs', ascending=True)
        .head(top_n)[cols_ex]
        .reset_index(drop=True)
    )

    top_lake_nz_excel0_df = (
        compare_df[
            (compare_df['commission_monthly_lake'].abs() >= exact_abs_tol)
            & (compare_df['commission_monthly_excel'].abs() < exact_abs_tol)
            & (~compare_df['is_exact_match'])
        ]
        .sort_values(by='commission_monthly_lake', key=lambda s: s.abs(), ascending=False)
        .head(top_n)[cols_ex]
        .reset_index(drop=True)
    )

    top_excel_nz_nolake_df = (
        compare_df[
            (compare_df['merge_status'] == 'right_only')
            & (compare_df['commission_monthly_excel'].abs() >= exact_abs_tol)
        ]
        .sort_values('commission_monthly_excel', ascending=False)
        .head(top_n)[cols_ex]
        .reset_index(drop=True)
    )

    # Примеры, похожие на округление / формат
    rounding_examples_df = (
        mismatches[mismatches['mismatch_reason'].isin([
            'near_rounding_abs_lt_1', 'near_rounding_pct_lt_1', 'suspect_integer_round',
            'suspect_scale_100x_or_0_01x',
        ])]
        .sort_values(by='delta_abs', key=lambda s: s.abs(), ascending=False)
        .head(30)[cols_ex]
        .reset_index(drop=True)
    )

    # Если rounding-примеров мало — добавить near-miss с |delta| < 10
    if len(rounding_examples_df) < 10:
        extra = (
            mismatches[
                (mismatches['delta_abs'].abs() < 10)
                & (~mismatches['is_exact_match'])
            ]
            .sort_values(by='delta_abs', key=lambda s: s.abs(), ascending=False)
            .head(20)[cols_ex]
        )
        rounding_examples_df = (
            pd.concat([rounding_examples_df, extra], ignore_index=True)
            .drop_duplicates(subset=['inn_key', 'agr_id_key'])
            .head(30)
        )

    return {
        'month_label': month_label,
        'resolved_excel_cols': resolved,
        'lake_df': lake_df,
        'excel_df': excel_df,
        'compare_df': compare_df,
        'summary_df': summary_df,
        'reason_stats_df': reason_stats_df,
        'top_delta_df': top_delta_df,
        'top_pos_delta_df': top_pos_delta_df,
        'top_neg_delta_df': top_neg_delta_df,
        'top_lake_nz_excel0_df': top_lake_nz_excel0_df,
        'top_excel_nz_nolake_df': top_excel_nz_nolake_df,
        'rounding_examples_df': rounding_examples_df,
    }




def load_sa_perimeter_month(month_label, month_start, month_end):
    """SA-контур как в 01_07_acq_dash (acq_class=SA, terms type P)."""
    sql = f"""
    select distinct
      cast(a.abs_agr_id as string) as agr_id_raw,
      regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_raw
    from {agreements_table} a
    join {companies_table} c
      on c.n_cmp = a.n_cmp_client
    where upper(trim(cast(a.acq_class as string))) = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
      and coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
      and c.c_inn is not null
      and exists (
          select 1
          from {agr_terms_table} t
          where cast(t.n_agr as string) = cast(a.n_agr as string)
            and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
            and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
            and upper(trim(cast(t.cf_ter_type as string))) = 'P'
            and coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
      )
    """
    df = run_sql(sql, step_name=f'sa_perimeter_{month_label}')
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=['month_label', 'inn_key', 'agr_id_key'])
    out = pd.DataFrame({
        'month_label': month_label,
        'inn_key': df['inn_raw'].apply(normalize_inn_q1),
        'agr_id_key': df['agr_id_raw'].apply(normalize_agr_q1),
    }).dropna(subset=['inn_key', 'agr_id_key']).drop_duplicates()
    return out


def load_rent_grain_qc_month(month_label, month_start, month_end):
    """Зерно rent: дни d_rent на c_nmrc и число мерчантов на agr_id (unique mapping)."""
    cte = build_mapping_cte(month_start, month_end)
    sql = f"""
    {cte}
    , map_stats as (
        select
            c_nmrc,
            d_rent_dt,
            n_amt_num,
            count(distinct case when inn_key is not null and agr_id_key is not null
                then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
        from mapped_raw
        group by c_nmrc, d_rent_dt, n_amt_num
    ), mapped_unique as (
        select
            mr.c_nmrc,
            mr.d_rent_dt,
            mr.n_amt_num,
            max(mr.inn_key) as inn_key,
            max(mr.agr_id_key) as agr_id_key
        from mapped_raw mr
        join map_stats ms
          on ms.c_nmrc = mr.c_nmrc
         and ms.d_rent_dt = mr.d_rent_dt
         and ms.n_amt_num = mr.n_amt_num
        where ms.valid_key_cnt = 1
          and mr.inn_key is not null
          and mr.agr_id_key is not null
        group by mr.c_nmrc, mr.d_rent_dt, mr.n_amt_num
    ), nmrc_days as (
        select
            c_nmrc,
            count(distinct d_rent_dt) as rent_day_cnt,
            sum(n_amt_num) as n_amt_sum,
            max(n_amt_num) as n_amt_max,
            min(n_amt_num) as n_amt_min
        from mapped_unique
        group by c_nmrc
    ), agr_merchants as (
        select
            inn_key,
            agr_id_key,
            count(distinct c_nmrc) as nmrc_cnt,
            sum(n_amt_num) as n_amt_sum
        from mapped_unique
        group by inn_key, agr_id_key
    )
    select
        '{month_label}' as month_label,
        (select count(*) from mapped_unique) as unique_rent_rows,
        (select count(distinct c_nmrc) from mapped_unique) as distinct_nmrc_cnt,
        (select count(distinct concat_ws('|', inn_key, agr_id_key)) from mapped_unique) as distinct_agr_key_cnt,
        (select avg(rent_day_cnt) from nmrc_days) as avg_rent_days_per_nmrc,
        (select max(rent_day_cnt) from nmrc_days) as max_rent_days_per_nmrc,
        (select sum(case when rent_day_cnt > 1 then 1 else 0 end) from nmrc_days) as nmrc_with_multi_days_cnt,
        (select sum(case when rent_day_cnt > 1 then n_amt_sum else 0 end) from nmrc_days) as n_amt_multi_day_nmrc,
        (select sum(case when rent_day_cnt > 1 and abs(n_amt_max - n_amt_min) < 0.01
            then n_amt_sum - n_amt_max else 0 end) from nmrc_days) as n_amt_suspect_full_month_dup,
        (select avg(nmrc_cnt) from agr_merchants) as avg_nmrc_per_agr,
        (select max(nmrc_cnt) from agr_merchants) as max_nmrc_per_agr,
        (select sum(case when nmrc_cnt > 1 then 1 else 0 end) from agr_merchants) as agr_with_multi_nmrc_cnt,
        (select sum(case when nmrc_cnt > 1 then n_amt_sum else 0 end) from agr_merchants) as n_amt_multi_nmrc_agr
    """
    df = run_sql(sql, step_name=f'rent_grain_qc_{month_label}')
    if df is None or len(df) == 0:
        return pd.DataFrame([{'month_label': month_label}])
    return df


def load_ambiguous_detail_month(month_label, month_start, month_end, limit=100):
    cte = build_mapping_cte(month_start, month_end)
    sql = f"""
    {cte}
    , map_stats as (
        select
            c_nmrc,
            d_rent_dt,
            n_amt_num,
            count(distinct case when inn_key is not null and agr_id_key is not null
                then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
        from mapped_raw
        group by c_nmrc, d_rent_dt, n_amt_num
    ), amb as (
        select c_nmrc, d_rent_dt, n_amt_num, valid_key_cnt
        from map_stats
        where valid_key_cnt > 1
    )
    select
        '{month_label}' as month_label,
        a.c_nmrc,
        a.d_rent_dt,
        a.n_amt_num,
        a.valid_key_cnt,
        mr.inn_key,
        mr.agr_id_key
    from amb a
    join mapped_raw mr
      on mr.c_nmrc = a.c_nmrc
     and mr.d_rent_dt = a.d_rent_dt
     and mr.n_amt_num = a.n_amt_num
    where mr.inn_key is not null
      and mr.agr_id_key is not null
    order by a.n_amt_num desc, a.c_nmrc, mr.agr_id_key
    limit {int(limit)}
    """
    df = run_sql(sql, step_name=f'ambiguous_detail_{month_label}')
    if df is None:
        return pd.DataFrame(columns=[
            'month_label', 'c_nmrc', 'd_rent_dt', 'n_amt_num', 'valid_key_cnt', 'inn_key', 'agr_id_key'
        ])
    df['inn_key'] = df['inn_key'].apply(normalize_inn_q1)
    df['agr_id_key'] = df['agr_id_key'].apply(normalize_agr_q1)
    return df


print('compare helpers ready')



## 2b) Mapping coverage Jan–Jun (no / unique / ambiguous)

Помесячно после дедупа `c_nmrc + d_rent`:
- **no** (`valid_key_cnt = 0`) — нет связки `inn+agr_id`;
- **unique** (`= 1`) — попадает в сверку/витрину;
- **ambiguous** (`> 1`) — несколько договоров на мерчанта, сейчас отбрасывается.

Критичность смотрите по `*_n_amt_pct` (доля ₽), не только по числу строк.


In [ ]:
if not access_ok:
    raise RuntimeError('Нет доступа к MPOS_RENT / scd1_mrc_pos_rent — coverage невозможен.')

coverage_rows = []
ambiguous_top_rows = []

for month_label in excel_reference_by_month:
    month_start, month_end = month_bounds(month_label)
    print('\n' + '-' * 80)
    print(f'MAPPING COVERAGE {month_label} | {month_start} .. {month_end}')
    print('-' * 80)

    cov_df = load_mapping_coverage_month(month_label, month_start, month_end)
    coverage_rows.append(cov_df)
    display(cov_df)

    amb_df = load_mapping_ambiguous_top_month(month_label, month_start, month_end, limit=50)
    ambiguous_top_rows.append(amb_df)
    if len(amb_df):
        print(f'[{month_label}] TOP ambiguous (by n_amt):')
        display(amb_df.head(20))
    else:
        print(f'[{month_label}] ambiguous rows: 0')

mapping_coverage_all_df = (
    pd.concat(coverage_rows, ignore_index=True) if coverage_rows else pd.DataFrame()
)
mapping_ambiguous_top_all_df = (
    pd.concat(ambiguous_top_rows, ignore_index=True) if ambiguous_top_rows else pd.DataFrame()
)

print('\n' + '=' * 80)
print('MAPPING COVERAGE ALL MONTHS')
print('=' * 80)
display(mapping_coverage_all_df[[
    'month_label',
    'rent_rows_after_dedup', 'total_n_amt',
    'no_mapping_rows', 'no_mapping_rows_pct', 'no_mapping_n_amt', 'no_mapping_n_amt_pct',
    'unique_mapping_rows', 'unique_mapping_rows_pct', 'unique_mapping_n_amt', 'unique_mapping_n_amt_pct',
    'ambiguous_mapping_rows', 'ambiguous_mapping_rows_pct', 'ambiguous_mapping_n_amt', 'ambiguous_mapping_n_amt_pct',
]])


## 3) Прогон по всем месяцам + примеры


In [ ]:
if not access_ok:
    raise RuntimeError('Нет доступа к MPOS_RENT / scd1_mrc_pos_rent — сверка невозможна.')

month_results = {}
summary_rows = []
reason_rows = []

for month_label, excel_path in excel_reference_by_month.items():
    header = int(excel_header_by_month.get(month_label, excel_header_default))
    print('\n' + '=' * 80)
    print(f'MONTH {month_label} | excel={excel_path} | header={header}')
    print('=' * 80)

    result = compare_month(month_label, excel_path, header)
    month_results[month_label] = result
    summary_rows.append(result['summary_df'])
    reason_rows.append(result['reason_stats_df'])

    print('\n--- 1) Сводка: % полного совпадения (нет в Lake => Excel должен быть 0) ---')
    display(result['summary_df'][[
        'month_label',
        'excel_key_cnt', 'lake_key_cnt', 'intersection_key_cnt',
        'full_exact_match_cnt', 'full_exact_match_rate_pct',
        'full_near_match_rate_pct', 'full_mismatch_cnt',
        'excel_total', 'lake_total', 'total_delta_abs',
        'mismatch_pos_delta_sum', 'mismatch_neg_delta_sum', 'mismatch_net_delta',
        'intersection_exact_match_rate_pct',
    ]])
    print(
        f"[{month_label}] full_exact_match_rate={result['summary_df'].loc[0, 'full_exact_match_rate_pct']:.2f}% | "
        f"net_delta={result['summary_df'].loc[0, 'mismatch_net_delta']:,.2f} | "
        f"pos={result['summary_df'].loc[0, 'mismatch_pos_delta_sum']:,.2f} | "
        f"neg={result['summary_df'].loc[0, 'mismatch_neg_delta_sum']:,.2f}"
    )

    print('\n--- 1b) Разбивка mismatch_reason (почему net != max single delta) ---')
    display(result['reason_stats_df'])

    print(f'\n--- 2) TOP-{top_n} |delta| (все ключи) ---')
    display(result['top_delta_df'])

    print(f'\n--- 2b) TOP-{top_n} Lake > Excel (положительные дельты) ---')
    display(result['top_pos_delta_df'])

    print(f'\n--- 2c) TOP-{top_n} Lake < Excel (компенсирующие отрицательные) ---')
    display(result['top_neg_delta_df'])

    print(f'\n--- 3) TOP-{top_n}: Lake != 0, Excel == 0 ---')
    display(result['top_lake_nz_excel0_df'])

    print(f'\n--- 4) TOP-{top_n}: Excel != 0, нет в Lake (ожидали 0) ---')
    display(result['top_excel_nz_nolake_df'])

    print('\n--- 5) Примеры похожие на округление / масштаб (запятая, 100x, |delta|<1) ---')
    display(result['rounding_examples_df'])

summary_all_df = pd.concat(summary_rows, ignore_index=True) if summary_rows else pd.DataFrame()
reason_stats_all_df = pd.concat(reason_rows, ignore_index=True) if reason_rows else pd.DataFrame()
print('\n' + '=' * 80)
print('SUMMARY ALL MONTHS — full match % + delta compensation')
print('=' * 80)
display(summary_all_df[[
    'month_label', 'excel_key_cnt', 'lake_key_cnt',
    'full_exact_match_rate_pct', 'full_near_match_rate_pct',
    'excel_total', 'lake_total', 'total_delta_abs',
    'mismatch_pos_delta_sum', 'mismatch_neg_delta_sum', 'mismatch_net_delta',
    'intersection_exact_match_rate_pct',
]])


## 3b) Диагностика января 2026 — почему Excel total «ломается»

Цель: понять аномалию января (`excel_total` ≫ Lake, низкий full_exact_match).
Проверяем header, выбор колонки комиссии, масштаб сумм, TOP Excel≠0 без Lake.


In [ ]:
# === January diagnostics ===
jan_label = '2026-01'
jan_path = excel_reference_by_month[jan_label]
jan_header_cfg = int(excel_header_by_month.get(jan_label, excel_header_default))
jan_month_start, jan_month_end = month_bounds(jan_label)

print('January file:', jan_path)
print('Configured header:', jan_header_cfg)
print('Lake period:', jan_month_start, '..', jan_month_end)

# --- A) Header 0 vs 1: columns and resolved commission ---
header_probe_rows = []
raw_by_header = {}

for hdr in [0, 1]:
    try:
        raw = pd.read_excel(jan_path, header=hdr)
    except Exception as exc:
        print(f'header={hdr}: FAILED to read: {type(exc).__name__}: {exc}')
        continue
    raw_by_header[hdr] = raw
    resolved = {k: pick_col_robust(raw.columns, v) for k, v in excel_col_map.items()}
    header_probe_rows.append({
        'header': hdr,
        'n_rows': len(raw),
        'n_cols': len(raw.columns),
        'inn_col': resolved.get('inn_col'),
        'agr_col': resolved.get('agr_col'),
        'comm_monthly_col': resolved.get('comm_monthly_col'),
        'tariff_col': resolved.get('tariff_col'),
        'is_configured_header': hdr == jan_header_cfg,
    })
    print(f'\n=== header={hdr} columns (first 40) ===')
    print(list(raw.columns)[:40])

header_probe_df = pd.DataFrame(header_probe_rows)
print('\n--- Resolved columns by header ---')
display(header_probe_df)

# --- B) Numeric column profile for configured header (+ alt header) ---
comm_candidates_extra = [
    'Комиссия в месяц', 'Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)',
    'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)',
    'Комиссия', 'commission_monthly', 'Комиссия за месяц',
    'Оборот', 'Сумма операций', 'trx_sum', 'ТО', 'ЧОД',
    'Комиссия эквайринга', 'Комиссия всего', 'Итого комиссия',
]


def _profile_numeric_columns(raw_df, header_val, top_n_cols=25):
    rows = []
    cols = list(raw_df.columns)
    # score all columns that look numeric after to_num_series
    for col in cols:
        s = to_num_series(raw_df[col])
        nn = int(s.notna().sum())
        if nn == 0:
            continue
        s2 = s.fillna(0)
        rows.append({
            'header': header_val,
            'column': str(col),
            'non_null_cnt': nn,
            'nonzero_cnt': int((s2.abs() >= 0.01).sum()),
            'sum': float(s2.sum()),
            'mean': float(s.mean()) if nn else np.nan,
            'p50': float(s.median()) if nn else np.nan,
            'p95': float(s.quantile(0.95)) if nn else np.nan,
            'max': float(s.max()) if nn else np.nan,
            'name_looks_commission': any(
                pick_col_robust([col], [c]) is not None for c in comm_candidates_extra
            ) or ('комисс' in str(col).lower()),
        })
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return out
    # prioritize commission-like and large sums
    out = out.sort_values(['name_looks_commission', 'sum'], ascending=[False, False]).reset_index(drop=True)
    return out.head(top_n_cols)


numeric_profiles = []
for hdr, raw in raw_by_header.items():
    prof = _profile_numeric_columns(raw, hdr)
    numeric_profiles.append(prof)
    print(f'\n--- Numeric column profile header={hdr} (TOP by commission-like / sum) ---')
    display(prof)

numeric_profile_all_df = pd.concat(numeric_profiles, ignore_index=True) if numeric_profiles else pd.DataFrame()

# --- C) Current pipeline load vs alternatives ---
lake_jan_df = load_lake_month(jan_label, jan_month_start, jan_month_end)
lake_jan_total = float(lake_jan_df['commission_monthly_lake'].fillna(0).sum()) if len(lake_jan_df) else 0.0
print(f'\nLake January total (unique mapping): {lake_jan_total:,.2f} | keys={len(lake_jan_df):,}')

alt_load_rows = []
for hdr in sorted(raw_by_header):
    raw = raw_by_header[hdr]
    resolved = {k: pick_col_robust(raw.columns, v) for k, v in excel_col_map.items()}
    if resolved.get('inn_col') is None or resolved.get('agr_col') is None or resolved.get('comm_monthly_col') is None:
        alt_load_rows.append({
            'header': hdr,
            'status': 'missing_required_cols',
            'resolved_comm_col': resolved.get('comm_monthly_col'),
        })
        continue
    tmp = raw.copy()
    tmp['inn_key'] = tmp[resolved['inn_col']].apply(normalize_inn_q1)
    tmp['agr_id_key'] = tmp[resolved['agr_col']].apply(normalize_agr_q1)
    tmp['commission_monthly_excel'] = to_num_series(tmp[resolved['comm_monthly_col']])
    agg = (
        tmp.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg(commission_monthly_excel=('commission_monthly_excel', 'max'))
    )
    excel_total = float(agg['commission_monthly_excel'].fillna(0).sum())
    excel_nz = agg[agg['commission_monthly_excel'].fillna(0).abs() >= 0.01]
    # match vs lake on keys
    m = lake_jan_df.merge(agg, on=['inn_key', 'agr_id_key'], how='outer', indicator=True)
    m['commission_monthly_lake'] = m['commission_monthly_lake'].fillna(0.0)
    m['commission_monthly_excel'] = m['commission_monthly_excel'].fillna(0.0)
    m['expected_lake'] = np.where(m['_merge'].isin(['both', 'left_only']), m['commission_monthly_lake'], 0.0)
    m['delta'] = m['expected_lake'] - m['commission_monthly_excel']
    excel_univ = m[m['_merge'].isin(['both', 'right_only'])]
    exact = int((excel_univ['delta'].abs() < exact_abs_tol).sum()) if len(excel_univ) else 0
    alt_load_rows.append({
        'header': hdr,
        'status': 'ok',
        'is_configured_header': hdr == jan_header_cfg,
        'resolved_comm_col': resolved['comm_monthly_col'],
        'excel_key_cnt': int(len(agg)),
        'excel_nz_key_cnt': int(len(excel_nz)),
        'excel_total': excel_total,
        'lake_total': lake_jan_total,
        'excel_to_lake_ratio': excel_total / lake_jan_total if lake_jan_total else np.nan,
        'full_exact_match_rate_pct': exact / len(excel_univ) * 100.0 if len(excel_univ) else np.nan,
        'abs_net_delta': abs(float(m['delta'].sum())),
        'closer_to_lake_than_10pct': abs(excel_total - lake_jan_total) / lake_jan_total < 0.10 if lake_jan_total else False,
        'scale_like_other_months_3_5m': 2_500_000 <= excel_total <= 5_500_000,
    })

alt_header_compare_df = pd.DataFrame(alt_load_rows)
print('\n--- Header alternatives: Excel total vs Lake + match % ---')
display(alt_header_compare_df)

# --- D) If configured load looks wrong: try ALL commission-like columns at cfg header ---
cfg_raw = raw_by_header.get(jan_header_cfg)
column_trial_rows = []
if cfg_raw is not None:
    resolved_cfg = {k: pick_col_robust(cfg_raw.columns, v) for k, v in excel_col_map.items()}
    inn_c = resolved_cfg.get('inn_col')
    agr_c = resolved_cfg.get('agr_col')
    # candidates = configured + any column with комисс in name + numeric profile commission-like
    cand_cols = []
    if resolved_cfg.get('comm_monthly_col'):
        cand_cols.append(resolved_cfg['comm_monthly_col'])
    for col in cfg_raw.columns:
        name = str(col).lower()
        if 'комисс' in name or 'commission' in name or 'месяц' in name:
            cand_cols.append(col)
    cand_cols = list(dict.fromkeys(cand_cols))

    if inn_c and agr_c:
        for col in cand_cols:
            tmp = cfg_raw.copy()
            tmp['inn_key'] = tmp[inn_c].apply(normalize_inn_q1)
            tmp['agr_id_key'] = tmp[agr_c].apply(normalize_agr_q1)
            tmp['v'] = to_num_series(tmp[col])
            agg = (
                tmp.dropna(subset=['inn_key', 'agr_id_key'])
                .groupby(['inn_key', 'agr_id_key'], as_index=False)
                .agg(v=('v', 'max'))
            )
            total = float(agg['v'].fillna(0).sum())
            m = lake_jan_df.merge(agg, on=['inn_key', 'agr_id_key'], how='inner')
            if len(m):
                m['commission_monthly_lake'] = pd.to_numeric(m['commission_monthly_lake'], errors='coerce').fillna(0)
                m['v'] = pd.to_numeric(m['v'], errors='coerce').fillna(0)
                exact_both = int(((m['commission_monthly_lake'] - m['v']).abs() < exact_abs_tol).sum())
                inter_exact_pct = exact_both / len(m) * 100.0
            else:
                inter_exact_pct = np.nan
            column_trial_rows.append({
                'header': jan_header_cfg,
                'column': str(col),
                'is_current_pick': col == resolved_cfg.get('comm_monthly_col'),
                'excel_total_max_by_key': total,
                'intersection_keys': int(len(m)),
                'intersection_exact_match_rate_pct': inter_exact_pct,
                'abs_diff_vs_lake_total': abs(total - lake_jan_total),
                'ratio_vs_lake': total / lake_jan_total if lake_jan_total else np.nan,
            })

column_trial_df = pd.DataFrame(column_trial_rows)
if len(column_trial_df):
    column_trial_df = column_trial_df.sort_values(
        ['intersection_exact_match_rate_pct', 'abs_diff_vs_lake_total'],
        ascending=[False, True],
    ).reset_index(drop=True)
    print('\n--- Column trials at configured header (best = high intersection exact / close lake total) ---')
    display(column_trial_df)

# --- E) Distribution of currently selected commission ---
excel_jan_df, resolved_jan = load_excel_month(jan_label, jan_path, jan_header_cfg)
s = excel_jan_df['commission_monthly_excel'].fillna(0)
dist_df = pd.DataFrame([{
    'resolved_comm_col': resolved_jan.get('comm_monthly_col'),
    'keys': len(excel_jan_df),
    'sum': float(s.sum()),
    'zero_cnt': int((s.abs() < 0.01).sum()),
    'nonzero_cnt': int((s.abs() >= 0.01).sum()),
    'mean_nz': float(s[s.abs() >= 0.01].mean()) if (s.abs() >= 0.01).any() else np.nan,
    'p50_nz': float(s[s.abs() >= 0.01].median()) if (s.abs() >= 0.01).any() else np.nan,
    'p95_nz': float(s[s.abs() >= 0.01].quantile(0.95)) if (s.abs() >= 0.01).any() else np.nan,
    'max': float(s.max()),
    'keys_gt_100k': int((s >= 100_000).sum()),
    'sum_gt_100k': float(s[s >= 100_000].sum()),
    'keys_gt_1m': int((s >= 1_000_000).sum()),
    'sum_gt_1m': float(s[s >= 1_000_000].sum()),
}])
print('\n--- Current pipeline Excel commission distribution ---')
display(dist_df)

top_excel_jan_df = (
    excel_jan_df.sort_values('commission_monthly_excel', ascending=False)
    .head(30)
    [['inn_key', 'agr_id_key', 'commission_monthly_excel', 'tariff_name_excel']]
    .reset_index(drop=True)
)
print('TOP-30 Excel January amounts (current pick):')
display(top_excel_jan_df)

# --- F) From month_results if available: mismatch focus ---
jan_mismatch_focus_df = pd.DataFrame()
jan_top_excel_nz_nolake_df = pd.DataFrame()
if 'month_results' in globals() and jan_label in month_results:
    jr = month_results[jan_label]
    print('\n--- From month_results[2026-01]: summary + reasons ---')
    display(jr['summary_df'])
    display(jr['reason_stats_df'])
    jan_top_excel_nz_nolake_df = jr['top_excel_nz_nolake_df'].copy()
    print('TOP Excel≠0 missing in Lake (ожидали 0 по правилу — но Excel огромный):')
    display(jan_top_excel_nz_nolake_df)

    # Are TOP excel amounts maybe turnover? compare to lake on same keys
    cmp = jr['compare_df']
    focus = cmp[
        (cmp['merge_status'] == 'right_only')
        & (cmp['commission_monthly_excel'].abs() >= 0.01)
    ].sort_values('commission_monthly_excel', ascending=False).head(50)
    jan_mismatch_focus_df = focus[[
        'inn_key', 'agr_id_key', 'commission_monthly_excel', 'tariff_name_excel', 'mismatch_reason'
    ]].copy()
else:
    print('\n[info] month_results нет — прогон секции 3 даст TOP mismatch из пайплайна.')

# --- G) Verdict hints ---
verdict_hints = []
if len(alt_header_compare_df):
    best_hdr = alt_header_compare_df.sort_values(
        ['full_exact_match_rate_pct', 'abs_net_delta'],
        ascending=[False, True],
    ).iloc[0]
    if (not bool(best_hdr.get('is_configured_header', False))) and pd.notna(best_hdr.get('full_exact_match_rate_pct')):
        if float(best_hdr['full_exact_match_rate_pct']) > 90:
            verdict_hints.append(
                f"Похоже, нужен header={int(best_hdr['header'])} "
                f"(full_exact≈{best_hdr['full_exact_match_rate_pct']:.1f}%, excel_total≈{best_hdr['excel_total']:,.0f})"
            )
if len(column_trial_df):
    best_col = column_trial_df.iloc[0]
    if (not bool(best_col.get('is_current_pick', False))) and pd.notna(best_col.get('intersection_exact_match_rate_pct')):
        if float(best_col['intersection_exact_match_rate_pct']) >= 90:
            verdict_hints.append(
                f"Лучшая колонка комиссии: '{best_col['column']}' "
                f"(intersection exact≈{best_col['intersection_exact_match_rate_pct']:.1f}%, total≈{best_col['excel_total_max_by_key']:,.0f})"
            )
if float(dist_df.loc[0, 'sum']) > 10_000_000:
    verdict_hints.append(
        'Текущая сумма Excel > 10M — вероятно выбрана не месячная комиссия (оборот / другая метрика).'
    )
if float(dist_df.loc[0, 'keys_gt_1m']) > 0:
    verdict_hints.append(
        f"Есть ключи с комиссией > 1M (n={int(dist_df.loc[0, 'keys_gt_1m'])}) — проверьте единицы/колонку."
    )
if not verdict_hints:
    verdict_hints.append('Явных авто-подсказок нет — смотрите column_trial_df и numeric_profile вручную.')

jan_verdict_df = pd.DataFrame({'hint': verdict_hints})
print('\n=== JANUARY DIAGNOSIS HINTS ===')
display(jan_verdict_df)

# keep for export
january_diag = {
    'header_probe_df': header_probe_df,
    'numeric_profile_all_df': numeric_profile_all_df,
    'alt_header_compare_df': alt_header_compare_df,
    'column_trial_df': column_trial_df,
    'dist_df': dist_df,
    'top_excel_jan_df': top_excel_jan_df,
    'jan_mismatch_focus_df': jan_mismatch_focus_df,
    'jan_verdict_df': jan_verdict_df,
}
print('january_diag ready')


## 4) Расширенные проверки совпадения MPOS_RENT vs Excel

Чеклист оценки:
1. маппинг ambiguous/no vs Excel≠0;
2. зерно `d_rent` / несколько мерчантов на договор;
3. семантика Excel=0 vs нет в Lake + gross vs net delta;
4. дубли ключей в Excel;
5. match % только на SA-контуре (как витрина);
6. стабильность метрик по месяцам Jan–Jun.


In [ ]:
if not access_ok:
    raise RuntimeError('Нет доступа к таблице — расширенный QC невозможен.')
if 'month_results' not in globals() or not month_results:
    raise RuntimeError('Сначала выполните секцию 3 (прогон по месяцам).')

extended_qc_rows = []
excel_dup_rows = []
grain_qc_rows = []
sa_match_rows = []
ambiguous_vs_excel_frames = []
semantic_detail_rows = []

for month_label, result in month_results.items():
    month_start, month_end = month_bounds(month_label)
    compare_df = result['compare_df'].copy()
    excel_df = result['excel_df'].copy()
    summary = result['summary_df'].iloc[0]

    print('\n' + '=' * 80)
    print(f'EXTENDED QC {month_label}')
    print('=' * 80)

    # --- 3) Semantics + gross delta ---
    excel_univ = compare_df[compare_df['in_excel']].copy()
    excel_nz = excel_univ[excel_univ['commission_monthly_excel'].abs() >= exact_abs_tol]
    lake_nz_excel0 = compare_df[
        (compare_df['commission_monthly_lake'].abs() >= exact_abs_tol)
        & (compare_df['commission_monthly_excel'].abs() < exact_abs_tol)
    ]
    excel_nz_nolake = compare_df[
        (compare_df['merge_status'] == 'right_only')
        & (compare_df['commission_monthly_excel'].abs() >= exact_abs_tol)
    ]
    mismatches = compare_df[~compare_df['is_exact_match']].copy()
    gross_abs_delta = float(mismatches['delta_abs'].abs().sum()) if len(mismatches) else 0.0
    net_delta = float(summary['mismatch_net_delta'])
    excel_total = float(summary['excel_total'])

    semantic_row = {
        'month_label': month_label,
        'excel_key_cnt': int(summary['excel_key_cnt']),
        'excel_nz_key_cnt': int(len(excel_nz)),
        'full_exact_match_rate_pct': float(summary['full_exact_match_rate_pct']),
        'excel_nz_nolake_cnt': int(len(excel_nz_nolake)),
        'excel_nz_nolake_amt': float(excel_nz_nolake['commission_monthly_excel'].sum()) if len(excel_nz_nolake) else 0.0,
        'excel_nz_nolake_pct_of_excel_nz': (
            len(excel_nz_nolake) / len(excel_nz) * 100.0 if len(excel_nz) else np.nan
        ),
        'lake_nz_excel0_cnt': int(len(lake_nz_excel0)),
        'lake_nz_excel0_amt': float(lake_nz_excel0['commission_monthly_lake'].sum()) if len(lake_nz_excel0) else 0.0,
        'gross_abs_delta': gross_abs_delta,
        'net_delta': net_delta,
        'gross_to_net_ratio': (gross_abs_delta / abs(net_delta) if abs(net_delta) > exact_abs_tol else np.nan),
        'net_delta_pct_of_excel_total': (
            abs(net_delta) / abs(excel_total) * 100.0 if excel_total else np.nan
        ),
        'gross_abs_delta_pct_of_excel_total': (
            gross_abs_delta / abs(excel_total) * 100.0 if excel_total else np.nan
        ),
    }
    semantic_detail_rows.append(semantic_row)
    print('--- Semantics: Excel=0 vs нет в Lake + gross/net ---')
    display(pd.DataFrame([semantic_row]))

    # --- 4) Excel duplicate keys ---
    # excel_df already aggregated; check raw if available via reload light stats from excel_df duplicates impossible
    # Re-read raw for dup check
    excel_path = excel_reference_by_month[month_label]
    header = int(excel_header_by_month.get(month_label, excel_header_default))
    ex_raw = pd.read_excel(excel_path, header=header)
    resolved = {k: pick_col_robust(ex_raw.columns, v) for k, v in excel_col_map.items()}
    tmp = ex_raw.copy()
    tmp['inn_key'] = tmp[resolved['inn_col']].apply(normalize_inn_q1)
    tmp['agr_id_key'] = tmp[resolved['agr_col']].apply(normalize_agr_q1)
    tmp['commission_monthly_excel'] = to_num_series(tmp[resolved['comm_monthly_col']])
    tmp = tmp.dropna(subset=['inn_key', 'agr_id_key'])
    dup = (
        tmp.groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg(
            row_cnt=('commission_monthly_excel', 'size'),
            comm_min=('commission_monthly_excel', 'min'),
            comm_max=('commission_monthly_excel', 'max'),
            comm_sum=('commission_monthly_excel', 'sum'),
            comm_max_used=('commission_monthly_excel', 'max'),
        )
    )
    dup = dup[dup['row_cnt'] > 1].copy()
    dup['month_label'] = month_label
    dup['conflict_values'] = (dup['comm_max'] - dup['comm_min']).abs() >= exact_abs_tol
    excel_dup_rows.append(dup)
    print(f'--- Excel duplicate inn+agr keys: {len(dup):,} (conflicts={int(dup["conflict_values"].sum()) if len(dup) else 0}) ---')
    display(dup.sort_values('row_cnt', ascending=False).head(20))

    # --- 2) Grain QC ---
    grain_df = load_rent_grain_qc_month(month_label, month_start, month_end)
    grain_qc_rows.append(grain_df)
    print('--- Grain: d_rent days / merchants per agr ---')
    display(grain_df)

    # --- 5) SA contour match ---
    sa_df = load_sa_perimeter_month(month_label, month_start, month_end)
    if len(sa_df):
        cmp_sa = compare_df.merge(
            sa_df[['inn_key', 'agr_id_key']].drop_duplicates(),
            on=['inn_key', 'agr_id_key'],
            how='inner',
        )
    else:
        cmp_sa = compare_df.iloc[0:0].copy()
    sa_n = int(len(cmp_sa))
    sa_excel_univ = cmp_sa[cmp_sa['in_excel']] if sa_n else cmp_sa
    sa_excel_n = int(len(sa_excel_univ))
    sa_excel_exact = int(sa_excel_univ['is_exact_match'].sum()) if sa_excel_n else 0
    sa_row = {
        'month_label': month_label,
        'sa_key_cnt': int(len(sa_df)),
        'sa_in_compare_cnt': sa_n,
        'sa_excel_key_cnt': sa_excel_n,
        'sa_full_exact_match_cnt': sa_excel_exact,
        'sa_full_exact_match_rate_pct': sa_excel_exact / sa_excel_n * 100.0 if sa_excel_n else np.nan,
        'sa_excel_total': float(sa_excel_univ['commission_monthly_excel'].sum()) if sa_excel_n else 0.0,
        'sa_lake_total': float(sa_excel_univ['expected_lake'].sum()) if sa_excel_n else 0.0,
        'sa_net_delta': float(sa_excel_univ['delta_abs'].sum()) if sa_excel_n else 0.0,
    }
    sa_match_rows.append(sa_row)
    print('--- SA-contour match % (витринный контур) ---')
    display(pd.DataFrame([sa_row]))

    # --- 1) Ambiguous vs Excel nonzero ---
    amb_detail = load_ambiguous_detail_month(month_label, month_start, month_end, limit=200)
    if len(amb_detail):
        amb_keys = amb_detail[['inn_key', 'agr_id_key', 'n_amt_num', 'c_nmrc', 'valid_key_cnt']].drop_duplicates()
        amb_vs_ex = amb_keys.merge(
            excel_df[['inn_key', 'agr_id_key', 'commission_monthly_excel', 'tariff_name_excel']],
            on=['inn_key', 'agr_id_key'],
            how='left',
        )
        amb_vs_ex['month_label'] = month_label
        amb_vs_ex['excel_is_nz'] = amb_vs_ex['commission_monthly_excel'].fillna(0).abs() >= exact_abs_tol
        amb_vs_ex['in_excel'] = amb_vs_ex['commission_monthly_excel'].notna()
        ambiguous_vs_excel_frames.append(amb_vs_ex)
        crit = amb_vs_ex[amb_vs_ex['excel_is_nz']]
        print(
            f'--- Ambiguous mapping vs Excel: rows={len(amb_vs_ex):,}, '
            f'with Excel≠0={len(crit):,} (критично для витрины) ---'
        )
        display(
            amb_vs_ex.sort_values(['excel_is_nz', 'n_amt_num'], ascending=[False, False])
            .head(30)
        )
    else:
        print('--- Ambiguous mapping vs Excel: 0 ambiguous rows ---')

    extended_qc_rows.append({
        **semantic_row,
        **sa_row,
        'excel_dup_key_cnt': int(len(dup)),
        'excel_dup_conflict_cnt': int(dup['conflict_values'].sum()) if len(dup) else 0,
    })

# Aggregate frames
semantic_all_df = pd.DataFrame(semantic_detail_rows)
sa_match_all_df = pd.DataFrame(sa_match_rows)
excel_dup_all_df = pd.concat(excel_dup_rows, ignore_index=True) if excel_dup_rows else pd.DataFrame()
grain_qc_all_df = pd.concat(grain_qc_rows, ignore_index=True) if grain_qc_rows else pd.DataFrame()
ambiguous_vs_excel_all_df = (
    pd.concat(ambiguous_vs_excel_frames, ignore_index=True) if ambiguous_vs_excel_frames else pd.DataFrame()
)
extended_qc_all_df = pd.DataFrame(extended_qc_rows)

print('\n' + '=' * 80)
print('6) STABILITY Jan–Jun — ключевые метрики по месяцам')
print('=' * 80)

stability_df = summary_all_df.merge(
    semantic_all_df[[
        'month_label', 'excel_nz_nolake_cnt', 'excel_nz_nolake_amt',
        'lake_nz_excel0_cnt', 'lake_nz_excel0_amt',
        'gross_abs_delta', 'gross_to_net_ratio',
        'gross_abs_delta_pct_of_excel_total', 'net_delta_pct_of_excel_total',
    ]],
    on='month_label',
    how='left',
).merge(
    sa_match_all_df[[
        'month_label', 'sa_key_cnt', 'sa_full_exact_match_rate_pct', 'sa_net_delta',
    ]],
    on='month_label',
    how='left',
)

if 'mapping_coverage_all_df' in globals() and len(mapping_coverage_all_df):
    stability_df = stability_df.merge(
        mapping_coverage_all_df[[
            'month_label',
            'unique_mapping_n_amt_pct', 'no_mapping_n_amt_pct', 'ambiguous_mapping_n_amt_pct',
        ]],
        on='month_label',
        how='left',
    )

stability_view_cols = [
    c for c in [
        'month_label',
        'full_exact_match_rate_pct', 'full_near_match_rate_pct',
        'sa_full_exact_match_rate_pct',
        'unique_mapping_n_amt_pct', 'no_mapping_n_amt_pct', 'ambiguous_mapping_n_amt_pct',
        'excel_total', 'lake_total',
        'mismatch_pos_delta_sum', 'mismatch_neg_delta_sum', 'mismatch_net_delta',
        'gross_abs_delta', 'gross_to_net_ratio',
        'excel_nz_nolake_cnt', 'excel_nz_nolake_amt',
        'lake_nz_excel0_cnt', 'lake_nz_excel0_amt',
        'sa_key_cnt',
    ] if c in stability_df.columns
]
display(stability_df[stability_view_cols])

print('\n--- Verdict helpers (per month) ---')
verdict_rows = []
for _, r in stability_df.iterrows():
    flags = []
    fem = r.get('full_exact_match_rate_pct', np.nan)
    amb = r.get('ambiguous_mapping_n_amt_pct', np.nan)
    uniq = r.get('unique_mapping_n_amt_pct', np.nan)
    g2n = r.get('gross_to_net_ratio', np.nan)
    if pd.notna(fem) and fem < 95:
        flags.append('full_match<95%')
    if pd.notna(amb) and amb >= 5:
        flags.append('ambiguous_amt>=5%')
    if pd.notna(uniq) and uniq < 90:
        flags.append('unique_amt<90%')
    if pd.notna(g2n) and g2n >= 3:
        flags.append('strong_delta_compensation')
    if r.get('excel_nz_nolake_cnt', 0) and r['excel_nz_nolake_cnt'] > 0:
        flags.append('excel_nz_missing_in_lake')
    verdict_rows.append({
        'month_label': r['month_label'],
        'full_exact_match_rate_pct': fem,
        'sa_full_exact_match_rate_pct': r.get('sa_full_exact_match_rate_pct', np.nan),
        'flags': ', '.join(flags) if flags else 'OK_looks_stable',
    })
verdict_df = pd.DataFrame(verdict_rows)
display(verdict_df)

print('\nGrain QC all months:')
display(grain_qc_all_df)


## 5) Апрель: right-only Excel по тарифам

Как раньше: ключи только в Excel за апрель, разбивка по `Тариф` (count / %).


In [ ]:
april_label = '2026-04'
april_result = month_results.get(april_label)

if april_result is None:
    print('SKIP: нет результата за апрель')
    right_only_zero_sanity_df = pd.DataFrame()
    tariff_right_only_df = pd.DataFrame()
else:
    compare_april_key_df = april_result['compare_df']
    excel_april_key_df = april_result['excel_df']

    right_only_df = compare_april_key_df[compare_april_key_df['merge_status'] == 'right_only'].copy()
    right_only_n = int(len(right_only_df))
    excel_all_n = int(len(excel_april_key_df))

    right_only_df['tariff_name_excel'] = right_only_df['tariff_name_excel'].fillna('(пусто)')
    right_only_df.loc[
        right_only_df['tariff_name_excel'].astype(str).str.strip().isin({'', 'nan', 'None', 'null'}),
        'tariff_name_excel'
    ] = '(пусто)'

    zero_cnt = int((right_only_df['commission_monthly_excel'].fillna(0).abs() < 0.01).sum()) if right_only_n else 0
    nonzero_cnt = right_only_n - zero_cnt
    zero_pct = zero_cnt / right_only_n * 100.0 if right_only_n else np.nan

    right_only_zero_sanity_df = pd.DataFrame([{
        'month_label': april_label,
        'right_only_key_cnt': right_only_n,
        'zero_commission_cnt': zero_cnt,
        'nonzero_commission_cnt': nonzero_cnt,
        'zero_commission_pct': zero_pct,
    }])

    tariff_right_only_df = (
        right_only_df
        .groupby('tariff_name_excel', dropna=False, as_index=False)
        .agg(client_cnt=('agr_id_key', 'count'))
        .sort_values('client_cnt', ascending=False)
        .reset_index(drop=True)
    )
    if right_only_n:
        tariff_right_only_df['pct_of_right_only'] = tariff_right_only_df['client_cnt'] / right_only_n * 100.0
        tariff_right_only_df['pct_of_all_excel'] = tariff_right_only_df['client_cnt'] / excel_all_n * 100.0 if excel_all_n else np.nan
    else:
        tariff_right_only_df['pct_of_right_only'] = np.nan
        tariff_right_only_df['pct_of_all_excel'] = np.nan
    tariff_right_only_df['month_label'] = april_label
    tariff_right_only_df = tariff_right_only_df[
        ['month_label', 'tariff_name_excel', 'client_cnt', 'pct_of_right_only', 'pct_of_all_excel']
    ]

    print('Sanity: April right_only commission == 0?')
    display(right_only_zero_sanity_df)
    print(f'April right-only by tariff (sum={int(tariff_right_only_df["client_cnt"].sum()) if len(tariff_right_only_df) else 0:,}):')
    display(tariff_right_only_df)


## 6) Выгрузка отчёта (все месяцы)


In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)


def _sheet(name):
    return str(name)[:31]


with pd.ExcelWriter(output_report_path, engine='xlsxwriter') as writer:
    summary_all_df.to_excel(writer, sheet_name='summary_all', index=False)
    if 'january_diag' in globals() and isinstance(january_diag, dict):
        for sheet_name, df_name in [
            ('jan_header_probe', 'header_probe_df'),
            ('jan_numeric_profile', 'numeric_profile_all_df'),
            ('jan_header_compare', 'alt_header_compare_df'),
            ('jan_column_trial', 'column_trial_df'),
            ('jan_comm_dist', 'dist_df'),
            ('jan_top_excel', 'top_excel_jan_df'),
            ('jan_mismatch_focus', 'jan_mismatch_focus_df'),
            ('jan_verdict', 'jan_verdict_df'),
        ]:
            df = january_diag.get(df_name)
            if isinstance(df, pd.DataFrame) and len(df):
                df.to_excel(writer, sheet_name=_sheet(sheet_name), index=False)
    if 'reason_stats_all_df' in globals() and len(reason_stats_all_df):
        reason_stats_all_df.to_excel(writer, sheet_name='mismatch_reasons_all', index=False)
    if 'mapping_coverage_all_df' in globals() and len(mapping_coverage_all_df):
        mapping_coverage_all_df.to_excel(writer, sheet_name='mapping_coverage_all', index=False)
    if 'mapping_ambiguous_top_all_df' in globals() and len(mapping_ambiguous_top_all_df):
        mapping_ambiguous_top_all_df.to_excel(writer, sheet_name='mapping_ambiguous_top', index=False)
    if 'semantic_all_df' in globals() and len(semantic_all_df):
        semantic_all_df.to_excel(writer, sheet_name='semantic_gross_net', index=False)
    if 'sa_match_all_df' in globals() and len(sa_match_all_df):
        sa_match_all_df.to_excel(writer, sheet_name='sa_contour_match', index=False)
    if 'grain_qc_all_df' in globals() and len(grain_qc_all_df):
        grain_qc_all_df.to_excel(writer, sheet_name='rent_grain_qc', index=False)
    if 'excel_dup_all_df' in globals() and len(excel_dup_all_df):
        excel_dup_all_df.to_excel(writer, sheet_name='excel_dup_keys', index=False)
    if 'ambiguous_vs_excel_all_df' in globals() and len(ambiguous_vs_excel_all_df):
        ambiguous_vs_excel_all_df.to_excel(writer, sheet_name='ambiguous_vs_excel', index=False)
    if 'stability_df' in globals() and len(stability_df):
        stability_df.to_excel(writer, sheet_name='stability_all_months', index=False)
    if 'verdict_df' in globals() and len(verdict_df):
        verdict_df.to_excel(writer, sheet_name='verdict_flags', index=False)
    if 'right_only_zero_sanity_df' in globals() and len(right_only_zero_sanity_df):
        right_only_zero_sanity_df.to_excel(writer, sheet_name='apr_right_only_sanity', index=False)
    if 'tariff_right_only_df' in globals() and len(tariff_right_only_df):
        tariff_right_only_df.to_excel(writer, sheet_name='apr_tariff_right_only', index=False)

    for month_label, result in month_results.items():
        mm = month_label.replace('-', '')
        result['summary_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_summary'), index=False)
        result['reason_stats_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_reasons'), index=False)
        result['top_delta_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_top_delta'), index=False)
        result['top_pos_delta_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_top_pos'), index=False)
        result['top_neg_delta_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_top_neg'), index=False)
        result['top_lake_nz_excel0_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_lake_nz_excel0'), index=False)
        result['top_excel_nz_nolake_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_excel_nz_nolake'), index=False)
        result['rounding_examples_df'].to_excel(writer, sheet_name=_sheet(f'{mm}_rounding'), index=False)

print(f'Report saved: {output_report_path}')
print(f'months processed: {list(month_results)}')
if 'stability_df' in globals() and len(stability_df):
    cols = [c for c in [
        'month_label', 'full_exact_match_rate_pct', 'sa_full_exact_match_rate_pct',
        'unique_mapping_n_amt_pct', 'gross_to_net_ratio',
        'excel_nz_nolake_cnt', 'lake_nz_excel0_cnt',
    ] if c in stability_df.columns]
    display(stability_df[cols])
elif 'summary_all_df' in globals():
    display(summary_all_df)
